# Give Me Some Credit: classify delinquency with a Random Forest

Train a Random Forest on the prepared credit data and inspect its held-out classification results. Complete the companion data-preparation notebook before running this lesson.

### Topics
- Load prepared training and test CSV files
- Select matching features
- Train a Random Forest classifier
- Interpret imbalanced-class metrics and prediction errors

**Question for the lesson:** Does the model detect delinquency effectively, beyond achieving high overall accuracy?

> **Estimated time: 30 min**

## 1 · Import the tools

Use Pandas to load prepared tables, a Random Forest classifier to predict delinquency, and classification metrics and Matplotlib to inspect errors. The imported splitting function is not used because the files already contain separate training and test subsets.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

## 2 · Load the prepared training data

Run `give_me_some_credit.ipynb` first to generate `train.csv` and `test.csv` in this folder. Load the training export and inspect the columns, including the target and engineered repayment features.

In [ ]:
train_data = pd.read_csv('train.csv')
train_data

## 3 · Load the prepared test data

This file contains the previously held-out rows, transformed using the training-set medians and matching feature definitions. Keep these rows out of model fitting.

In [ ]:
test_data = pd.read_csv('test.csv')
test_data

## 4 · Select training features and target

Use `SeriousDlqin` as the target: `1` indicates serious delinquency and `0` its absence. Remove it from the predictors along with `Is_Outlier`, a training-only diagnostic flag. All remaining predictors must be numerical and free of unsupported missing values.

In [ ]:
X_train = train_data.drop(columns=['SeriousDlqin', "Is_Outlier"])
y_train = train_data['SeriousDlqin']

## 5 · Train a Random Forest classifier

Fit 100 trees using the prepared training rows. Each tree learns from a bootstrap sample and random feature subsets; the forest averages class probabilities to predict an outcome. `random_state=42` makes training reproducible. This example uses the default class weighting and prediction rule.

In [ ]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

## 6 · Prepare matching test inputs

Separate the test target from its predictors. Predictor names and order must match `X_train` exactly. The preparation notebook does not add `Is_Outlier` to test data, so only the target is dropped here. If you change either export, verify the two schemas before predicting.

In [ ]:
X_test = test_data.drop(columns=['SeriousDlqin'])
y_test = test_data['SeriousDlqin']

## 7 · Evaluate delinquency predictions

Predict the held-out rows and inspect accuracy together with the per-class report. For delinquency (`1`), precision measures how many flagged customers actually have the outcome; recall measures how many actual delinquent customers are detected. F1 balances the two, and support counts the true examples of each class.

Compare accuracy with the majority-class share. A high accuracy can hide poor delinquency recall when class `1` is rare.

In [ ]:
y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Classification Report:")
print(classification_rep)

## 8 · Interpret the confusion matrix

Rows are actual classes and columns are predicted classes. `Class 0` means no serious delinquency; `Class 1` means serious delinquency.

- Top left: correctly predicted non-delinquent customers.
- Top right: non-delinquent customers incorrectly flagged.
- Bottom left: delinquent customers the model missed.
- Bottom right: correctly detected delinquent customers.

### Pause & discuss

Which error is more costly in the intended lending workflow? How would that affect your choice of evaluation metric and decision threshold?

In [ ]:
cm = confusion_matrix(y_test, y_pred)

# 2. Plot the matrix visually
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(cmap=plt.cm.Blues)
plt.show()

## Key takeaways

- Fit the model on training rows and evaluate on held-out rows.
- Keep preprocessing and predictor schemas consistent across datasets.
- Inspect delinquency precision and recall alongside accuracy.
- Interpret errors in terms of the decisions they would affect.

### Try next

Compare these results with a majority-class baseline. As a separate experiment, use a validation subset of the training data to compare class weights or probability thresholds, then evaluate the chosen approach once on the test set.